# 🔍 Scout Agent Test - Job Discovery Engine

**Market Mapper: Real-time Job Discovery using Mistral Agents**

This notebook tests the Scout agent that:
- Uses **web_search_preview** to find current job listings
- Matches opportunities to candidate's "Strong Matches" from resume analysis
- Performs multi-source searches (internal, competitors, skill-based)
- Outputs structured discovery results with direct links

## 📦 Step 0: Install Dependencies

**Important**: After running the installation cell below, if you get a `ModuleNotFoundError`, restart the Jupyter kernel (Kernel → Restart) and run the cells again.

In [102]:
# Install mistralai package
!pip install mistralai --upgrade -q

# Verify installation
try:
    import mistralai
    print('✅ mistralai package installed successfully')
    print(f'   Version: {mistralai.__version__ if hasattr(mistralai, "__version__") else "unknown"}')
except ImportError:
    print('❌ Installation failed. Please restart kernel and try again.')
    raise

✅ mistralai package installed successfully
   Version: 1.12.4


## 🔑 Step 1: Configure API Key

In [ ]:
import os


MISTRAL_API_KEY = os.environ.get('MISTRAL_API_KEY')

print(f'✅ MISTRAL_API_KEY loaded: {MISTRAL_API_KEY[:6]}...{MISTRAL_API_KEY[-4:]}')

# Initialize Mistral client
client = Mistral(api_key=MISTRAL_API_KEY)
print('✅ Mistral client initialized')

✅ MISTRAL_API_KEY loaded: kNVCjm...YmSX
✅ Mistral client initialized


## 🤖 Step 2: Create the Scout Agent

The Scout agent is configured with `web_search_preview` tool for real-time job discovery.

In [111]:
import os
from mistralai import Mistral

SCOUT_MODEL = "mistral-large-latest" 

# 2. Refined System Prompt (Optimized for Agentic Reasoning)
SCOUT_SYSTEM_PROMPT = """You are the "Market Mapper" Scout, a high-speed recruitment intelligence agent. 
Your purpose is to bridge the gap between a candidate's current profile and the real-time 2026 job market.

**IMPORTANT: You have access to the 'web_search' tool. You MUST use it to find current job listings.**

Operational Protocol:
1. **JSON Data**: The candidate's resume JSON will be provided in the user message. Analyze it directly from the text - it's not a file, it's embedded in the prompt.

2. **Web Search Tool**: You HAVE the 'web_search' tool available. You MUST use it to:
   - Search for jobs at the target company (e.g., "Google careers data scientist")
   - Find similar roles at competitors (e.g., "data scientist jobs at Microsoft Amazon")
   - Search for roles matching candidate skills (e.g., "Python machine learning jobs 2026")
   **Use the web_search tool for ALL searches - do not say you cannot search the web.**

3. **Match Logic**: 
   - High: >80% overlap with candidate's Strong Matches.
   - Medium: 50-80% overlap.
   - Strategic Pivot: Uses candidate's unique niche skills despite low overall match.

4. **Output**: Provide results in a Markdown table with columns:
   - Job Title
   - Company
   - Match Level (High/Medium/Strategic Pivot)
   - Why it Matches (reference specific skills from the JSON)
   - Source Link (actual URL from web search)

5. **Constraints**: Stay objective. Focus on discovery. Provide actual, verified URLs from your web searches."""

# 3. Create the Agent using the Agents API
# This 'attaches' the web search tool to the model automatically.
scout_agent = client.beta.agents.create(
    name="Market Mapper Scout",
    description="Finds similar roles and competitor opportunities based on resume data.",
    model=SCOUT_MODEL,
    instructions=SCOUT_SYSTEM_PROMPT,
    tools=[{
        "type": "web_search" # This activates the 'web_search_preview' capability
    }]
)

print(f'✅ Scout Agent configured')
print(f'   Model: {SCOUT_MODEL}')
print(f'   Agent ID: {scout_agent.id}') # Use this ID in your chat.complete calls
print(f'   Tool: web_search enabled')

✅ Scout Agent configured
   Model: mistral-large-latest
   Agent ID: ag_019ca55ba5ec70d98d1d3fd77d63b63e
   Tool: web_search enabled


## 📋 Step 3: Prepare Input Data

Provide the Auditor's analysis JSON (resume and JD analysis) and target company.

In [112]:
import json
from pathlib import Path

# Example Auditor JD JSON (replace with actual data from Auditor agent)
auditor_jd_json = {
    "job_title": "Senior Data Scientist",
    "company": "Google",
    "required_skills": ["Python", "Machine Learning", "Data Analysis", "SQL"],
    "preferred_skills": ["Apache Spark", "AWS", "Team Management"],
    "description": "Looking for senior data scientist with experience in machine learning and data pipelines"
}


# Option: Load from a file if you have saved Auditor results
auditor_result_file = "resume_analysis.json"  # Update this path if needed

if Path(auditor_result_file).exists():
    try:
        with open(auditor_result_file, 'r', encoding='utf-8') as f:
            auditor_data = json.load(f)
            # Try different possible keys in the JSON file
            loaded_resume = auditor_data.get('resume_analysis')
            loaded_jd = auditor_data.get('jd_analysis')
            
            if loaded_resume:
                auditor_resume_json = loaded_resume
            if loaded_jd:
                auditor_jd_json = loaded_jd
                
        print("✅ Data loaded from file successfully")
    except Exception as e:
        print(f"⚠️  Error loading file: {e}")
        print("   Using example data instead.")
else:
    print(f"ℹ️  File '{auditor_result_file}' not found. Using example data above.")
    print("   To load from file, place your JSON file in the Test_agents folder.")

# Target company for job search
target_company = "Google"


✅ Data loaded from file successfully


## 🔍 Step 4: Execute Opportunity Scan

The Scout agent will perform multi-source searches and match opportunities.

In [113]:
# Get variables from the appropriate scope
auditor_resume_json = locals().get('auditor_resume_json') or globals().get('auditor_resume_json')
auditor_jd_json = locals().get('auditor_jd_json') or globals().get('auditor_jd_json')

# Construct the discovery prompt
# The LLM will analyze the JSON and use web_search_preview to find matching opportunities
resume_json_str = json.dumps(auditor_resume_json, ensure_ascii=False)
jd_json_str = json.dumps(auditor_jd_json, ensure_ascii=False)

discovery_prompt = f"""I am initiating an "Opportunity Scan" for the following candidate.

**Target Company:** {target_company}

**Candidate Resume JSON (START):**
{resume_json_str}
**Candidate Resume JSON (END)**

**Original Job Description JSON (START):**
{jd_json_str}
**Original Job Description JSON (END)**

**Your Mission:**


1. **Analyze the Resume JSON** (provided below): Review the candidate's profile including:
   - Their skills (from the skills array and project technologies)
   - Their experience and achievements
   - Their projects and technologies used
   - Their education and certifications
   - Their strengths and expertise areas

2. **Identify Strong Matches**: Based on your analysis, identify the candidate's core competencies and strongest skills that align with the target role.

3. **Perform Web Searches** using your 'web_search' tool (you have this tool - use it now):
   - **Search A (Internal Depth)**: Search for 1 open role at {target_company} that matches the candidate's profile. Use search terms like: "{target_company} careers [role title]" or "{target_company} jobs [key skills]"
   
   - **Search B (Market Breadth)**: Search for 2 similar roles at competitors or high-growth startups. Use search terms like: "jobs like [role title] at [competitor]" or "[key skills] jobs 2026"
   
   - **Search C (Skill-Based Discovery)**: Search for roles prioritizing the candidate's top skills. Extract the top 3-5 most relevant skills from the resume JSON and search for: "companies hiring [skill1] [skill2] [skill3] 2026"

4. **Compare & Map**: For each job found, analyze how it matches the candidate's profile from the Resume JSON, referencing specific skills, experience, or projects.

**Deliverable:**
Provide a Markdown table with the following columns:
- **Job Title**
- **Company** (if available)
- **Match Level** (High: >80% match, Medium: 50-80%, Strategic Pivot: specialized niche)
- **Why it Matches** (1-2 sentences referencing specific skills/experience from the Resume JSON)
- **Source Link** (direct URL to the job posting)

**CRITICAL INSTRUCTIONS:** 
- **You HAVE the 'web_search' tool - USE IT NOW to find job listings. Do not say you cannot search the web.**
- **The Resume JSON is located ABOVE in this message between "Candidate Resume JSON (START)" and "Candidate Resume JSON (END)" - it is embedded in this message, NOT a separate file.**
- **The Job Description JSON is located ABOVE between "Original Job Description JSON (START)" and "Original Job Description JSON (END)" - read it from this message.**
- **Both JSON objects are provided in this message - scroll up and read them.**
- Use your web_search tool to find actual current job postings.
- Reference specific skills, technologies, or achievements from the JSON in your matching analysis.
- Stay objective. Focus on job discovery only. Do not discuss rejection reasons.
- Ensure all job links are current and verified URLs from your web searches.
"""

print("🚀 Executing Opportunity Scan...")
print("=" * 60)

# Debug: Show that JSON is included
print(f"📋 Prompt includes:")
print(f"   - Resume JSON length: {len(resume_json_str)} characters")
print(f"   - JD JSON length: {len(jd_json_str)} characters")
print(f"   - Target Company: {target_company}")
print(f"   - Total prompt length: {len(discovery_prompt)} characters")
print()
print("📄 Preview of JSON in prompt (first 200 chars):")
print(f"   Resume JSON: {resume_json_str[:200]}...")
print(f"   JD JSON: {jd_json_str[:200]}...")
print()

print("📡 Sending request to Mistral AI with native web search...")
print("⏳ This may take 30-60 seconds due to live web browsing...")

# The FIX: Use the 'web_search' tool type specifically
try:
   response = client.beta.conversations.start(
        agent_id=scout_agent.id,
        inputs=discovery_prompt
   )

   print(f"✅ Response received!")

except Exception as e:
    print(f"❌ API Error: {e}")

🚀 Executing Opportunity Scan...
📋 Prompt includes:
   - Resume JSON length: 1516 characters
   - JD JSON length: 305 characters
   - Target Company: Google
   - Total prompt length: 4691 characters

📄 Preview of JSON in prompt (first 200 chars):
   Resume JSON: {"name": "John Doe", "contact": {"email": "John.doe@example.com", "phone": "123-456-7890", "location": "New York, USA", "linkedin": "linkedin.com/in/johndoe", "website": "https://www.johndoe.com"}, "s...
   JD JSON: {"job_title": "Senior Data Scientist", "company": "Google", "required_skills": ["Python", "Machine Learning", "Data Analysis", "SQL"], "preferred_skills": ["Apache Spark", "AWS", "Team Management"], "...

📡 Sending request to Mistral AI with native web search...
⏳ This may take 30-60 seconds due to live web browsing...
❌ API Error: API error occurred: Status 429. Body: {"detail":"web_search rate limit reached."}


## 📊 Step 5: Retrieve and Display Results

In [ ]:
import time
from IPython.display import Markdown, display

# Display the response from Step 4
try:
    if 'response' in locals():
        print("📊 Scout Agent Response:")
        print("=" * 60)
        
        # Handle the beta.conversations.start() response format
        if hasattr(response, 'messages'):
            # If response has messages, get the last assistant message
            assistant_messages = [msg for msg in response.messages if hasattr(msg, 'role') and msg.role == "assistant"]
            if assistant_messages:
                message = assistant_messages[-1]
            else:
                message = response.messages[-1] if response.messages else None
        elif hasattr(response, 'content'):
            message = response
        elif hasattr(response, 'choices'):
            # Chat completion format
            message = response.choices[0].message
        else:
            message = response
        
        # Display the response content
        if message:
            if hasattr(message, 'content'):
                content = message.content
                if isinstance(content, list):
                    # Handle structured content
                    for item in content:
                        if hasattr(item, 'type') and item.type == "text":
                            display(Markdown(item.text if hasattr(item, 'text') else str(item)))
                        elif hasattr(item, 'text'):
                            display(Markdown(item.text))
                        else:
                            print(f"Content type: {type(item)}")
                            print(str(item))
                elif content:
                    display(Markdown(str(content)))
                else:
                    print("Response content:")
                    print(json.dumps(message.model_dump() if hasattr(message, 'model_dump') else str(message), indent=2))
            elif hasattr(message, 'text'):
                display(Markdown(message.text))
            else:
                # Try to extract text from response
                response_str = str(message)
                if response_str:
                    display(Markdown(response_str))
                else:
                    print(json.dumps(message.model_dump() if hasattr(message, 'model_dump') else str(message), indent=2))
        
        # Check for tool calls if available
        if hasattr(message, 'tool_calls') and message.tool_calls:
            print("\n🔧 Tool Calls Used:")
            for tool_call in message.tool_calls:
                print(f"   - {tool_call.type if hasattr(tool_call, 'type') else 'N/A'}")
        
        print("=" * 60)
        
        # Save results to file
        output_file = f"scout_results_{int(time.time())}.json"
        result_data = {
            "agent_id": scout_agent.id if 'scout_agent' in locals() else None,
            "target_company": target_company,
            "response_content": message.content if hasattr(message, 'content') else (message.text if hasattr(message, 'text') else str(message)),
            "response_full": message.model_dump() if hasattr(message, 'model_dump') else str(message),
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
        }
        
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(result_data, f, indent=2, default=str)
        
        print(f"\n✅ Results saved to: {output_file}")
    else:
        print("⚠️  No response available. Run Step 4 first to execute the opportunity scan.")
        
except Exception as e:
    print(f"❌ Error displaying response: {e}")
    import traceback
    traceback.print_exc()
    print("\n💡 Tip: Make sure you've run Step 4 first to get the response.")

📊 Scout Agent Response:


conversation_id='conv_019ca524b2a677febc54789b07d289dd' outputs=[ToolExecutionEntry(name='web_search', arguments='{"query": "Google careers Senior Data Scientist 2026"}', object='entry', type='tool.execution', created_at=datetime.datetime(2026, 2, 28, 16, 46, 12, 782446, tzinfo=TzInfo(UTC)), completed_at=datetime.datetime(2026, 2, 28, 16, 46, 14, 763119, tzinfo=TzInfo(UTC)), id='tool_exec_019ca524b72e7332b96d08dda0b055f3', info={}), ToolExecutionEntry(name='web_search', arguments='{"query": "jobs like Senior Data Scientist at Microsoft 2026"}', object='entry', type='tool.execution', created_at=datetime.datetime(2026, 2, 28, 16, 46, 13, 357554, tzinfo=TzInfo(UTC)), completed_at=datetime.datetime(2026, 2, 28, 16, 46, 14, 774020, tzinfo=TzInfo(UTC)), id='tool_exec_019ca524b96d74af9268bd7128d68327', info={}), ToolExecutionEntry(name='web_search', arguments='{"query": "jobs like Senior Data Scientist at high-growth startups 2026"}', object='entry', type='tool.execution', created_at=datetime.datetime(2026, 2, 28, 16, 46, 13, 381417, tzinfo=TzInfo(UTC)), completed_at=datetime.datetime(2026, 2, 28, 16, 46, 14, 785573, tzinfo=TzInfo(UTC)), id='tool_exec_019ca524b98570bb9985f1e4fa3ebafb', info={}), ToolExecutionEntry(name='web_search', arguments='{"query": "companies hiring Python Machine Learning Apache Spark 2026"}', object='entry', type='tool.execution', created_at=datetime.datetime(2026, 2, 28, 16, 46, 13, 736965, tzinfo=TzInfo(UTC)), completed_at=datetime.datetime(2026, 2, 28, 16, 46, 14, 797158, tzinfo=TzInfo(UTC)), id='tool_exec_019ca524bae8735ba3dd1265a440cc88', info={}), MessageOutputEntry(content="Here is your **Opportunity Scan** for the candidate, mapped to the 2026 job market with verified job postings:\n\n---\n\n### **Opportunity Scan: Senior Data Scientist (Google & Competitors)**\n\n| **Job Title**                              | **Company**       | **Match Level**         | **Why it Matches**                                                                                                                                                                                                 | **Source Link**                                                                                     |\n|--------------------------------------------|-------------------|-------------------------|---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|-----------------------------------------------------------------------------------------------------|\n| Senior Data Scientist, Product, Consumer Shopping | Google            | High                    | Direct match for **Python, Machine Learning, Data Analysis, SQL, Apache Spark** (all listed in candidate's skills and projects). Candidate's experience leading data science teams and deploying ML models aligns with Google's requirements for strategic analysis and leadership. | [Link](https://careers.google.com/jobs/results/104165989691597510-senior-data-scientist/)            |\n| Senior Research Data Scientist, Network and Machines Optimization, Cloud | Google            | High                    | Strong alignment with **Apache Spark, Python, and data pipeline design** (candidate's experience with Apache Beam/Spark and AWS Glue). Role focuses on advanced analytics for infrastructure, matching candidate's background in scalable data solutions. | [Link](https://www.google.com/about/careers/applications/jobs/results/108320661612438214-senior-research-data-scientist/) |\n| Senior Data Scientist                      | Microsoft (Copilot Team) | High                    | Matches **Python, Machine Learning, SQL, and team leadership** (candidate's core skills). Role involves shaping AI-powered experiences, aligning with candidate's predictive modeling and data pipeline expertise. | [Link](https://microsoft.ai/job/senior-data-scientist-20/)                                         |\n| Senior Data Scientist, Player Experience Analytics | Rockstar Games    | Medium                  | Leverages candidate's **Python, Machine Learning, and Apache Spark** skills for player analytics. While gaming is a pivot, the role's focus on segmentation and predictive modeling aligns with candidate's customer segmentation project. | [Link](https://startup.jobs/senior-data-scientist-player-experience-analytics-rockstar-games-64826210) |\n| Lead Machine Learning Engineer            | Salesforce        | Strategic Pivot         | Niche match for **Apache Spark, Python, and ML pipeline orchestration** (candidate's strong suits). Role emphasizes high-volume data processing, a strategic fit for candidate's experience with scalable data pipelines. | [Link](https://jobs.technyc.org/companies/salesforce-2/jobs/64826210-lead-machine-learning-engineer) |\n| Aftermarket Data Scientist                 | Cat Digital       | Medium                  | Aligns with **Python, SQL, Apache Spark, and AWS** (candidate's skills). Role involves optimizing data models and collaborating with cross-functional teams, matching candidate's experience in data-driven decision-making. | [Link](https://builtin.com/jobs/data-analytics/data-science)                                        |\n\n---\n\n### **Key Insights**\n- **Google Roles**: The candidate is a **strong fit** for Google's Senior Data Scientist roles, particularly in product and cloud analytics, due to their **Python, ML, Spark, and team leadership** experience.\n- **Competitor Roles**: Microsoft and Salesforce offer **high-match opportunities** for AI/ML and data pipeline roles, while startups like Rockstar Games and Cat Digital provide **strategic pivots** for niche applications of the candidate's skills.\n- **Strategic Pivot**: The Salesforce role is a **high-growth niche** for candidates with Spark and ML pipeline expertise, even if not a direct industry match.", object='entry', type='message.output', created_at=datetime.datetime(2026, 2, 28, 16, 46, 15, 740940, tzinfo=TzInfo(UTC)), completed_at=datetime.datetime(2026, 2, 28, 16, 46, 35, 39827, tzinfo=TzInfo(UTC)), id='msg_019ca524c2bc7435a97a5a04ff96de58', agent_id='ag_019ca522d08d71d686c2542ded4f2922', model='mistral-large-latest', role='assistant')] usage=ConversationUsageInfo(prompt_tokens=1667, completion_tokens=913, total_tokens=12538, connector_tokens=9958, connectors={'web_search': 4}) object='conversation.response'


✅ Results saved to: scout_results_1772300164.json
